# Notebook 5 — Robustness evaluation

**Goal:** show that the inductive bias addresses the clinical problem — the **quality-aware MedVAE** should reconstruct degraded images better than the standard MedVAE.

**Experimental protocol:**

| Experiment | Description |
|---|---|
| **A — Real images** | Baseline vs cVAE on the `bad/medium/good` ARCADE images |
| **B — Synthetic degradations** | Artificially degrade the best images and compare the robustness of the two models |

**Metrics:** MSE, SSIM, PSNR

> How the score `c` is computed is set in `pipeline_config.py`. The checkpoint used matches the selected approach.

## 0. Configuration

In [ ]:
from pathlib import Path
from pipeline_config import APPROACH, get_approach_config, get_score_column, print_approach_summary

# ── Selected approach ──
print_approach_summary()

NOTEBOOK_DIR  = Path(".").resolve()
PROJECT_ROOT  = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

# Outputs organised by approach
OUTPUT_DIR  = Path(f"outputs_cvae/05_Evaluation_of_Robustness/approach_{APPROACH}")
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR  = OUTPUT_DIR / "tables"
for _d in [OUTPUT_DIR, FIGURES_DIR, TABLES_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

NB03_DIR      = Path("outputs_cvae/03_Inductive_Bias_Generation")
NB04_DIR      = Path(f"outputs_cvae/04_MedVAE_Architecture_Mod/approach_{APPROACH}")
LABELS_CSV    = NB03_DIR / "labels_quality.csv"     # NB3
CVAE_CKPT     = NB04_DIR / "checkpoints" / "quality_aware_vae.pt"   # NB4 (per approach)

DDCONFIG = {
    "double_z": True, "z_channels": 4, "resolution": 64,
    "in_channels": 1, "out_ch": 1, "ch": 128,
    "ch_mult": [1, 2, 4], "num_res_blocks": 2,
    "attn_resolutions": [], "dropout": 0.0,
}
EMBED_DIM = 4
IMG_SIZE  = DDCONFIG["resolution"]   # 64

N_TEST_PER_CLASS = 20    # images per class (bad/medium/good)
RANDOM_SEED      = 42

print(f"Checkpoint : {CVAE_CKPT}  (existe: {CVAE_CKPT.exists()})")
print(f"Sorties    : {OUTPUT_DIR}")

## 1. Imports

In [ ]:
import sys, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from scipy import stats
from scipy.ndimage import uniform_filter
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Import the standalone MedVAE (no need for jepa-adaptation any more)
from medvae_standalone import AutoencoderKL as AutoencoderKL2D
from medvae_standalone import DiagonalGaussianDistribution

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

## 2. Definition of QualityAwareAutoencoderKL (FiLM)

*(Redefined here to make this notebook standalone — same FiLM version as notebook 4)*

In [ ]:
class FiLMConditioner(nn.Module):
    """FiLM — Perez et al., AAAI 2018. Zero-init."""
    def __init__(self, channels_per_block: list, hidden_dim: int = 128):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
        )
        self.heads = nn.ModuleList()
        for ch in channels_per_block:
            head = nn.Linear(hidden_dim, 2 * ch)
            nn.init.zeros_(head.weight)
            nn.init.zeros_(head.bias)
            self.heads.append(head)

    def forward(self, c):
        emb = self.embed(c.view(-1, 1))
        return [
            (gb[:, :gb.shape[1]//2].unsqueeze(-1).unsqueeze(-1),
             gb[:, gb.shape[1]//2:].unsqueeze(-1).unsqueeze(-1))
            for gb in (head(emb) for head in self.heads)
        ]


class QualityAwareAutoencoderKL(nn.Module):
    """cVAE conditioned through FiLM (hooks on each ResnetBlock)."""
    def __init__(self, base_ae: AutoencoderKL2D):
        super().__init__()
        self.ae = base_ae
        self._blocks = list(self._iter_resnet_blocks())
        channels = [b.conv2.out_channels for b in self._blocks]
        self.film = FiLMConditioner(channels)

    def _iter_resnet_blocks(self):
        enc = self.ae.encoder
        for i_level in range(enc.num_resolutions):
            for i_block in range(enc.num_res_blocks):
                yield enc.down[i_level].block[i_block]
        yield enc.mid.block_1
        yield enc.mid.block_2
        dec = self.ae.decoder
        yield dec.mid.block_1
        yield dec.mid.block_2
        for i_level in reversed(range(dec.num_resolutions)):
            for i_block in range(dec.num_res_blocks + 1):
                yield dec.up[i_level].block[i_block]

    def _register_film_hooks(self, c):
        film_params = self.film(c)
        hooks = []
        for i, block in enumerate(self._blocks):
            gamma, beta = film_params[i]
            hook = block.register_forward_hook(
                lambda mod, inp, out, g=gamma, b=beta: (1 + g) * out + b
            )
            hooks.append(hook)
        return hooks

    def forward(self, x, c, sample_posterior=True):
        hooks = self._register_film_hooks(c)
        try:
            h = self.ae.encoder(x)
            moments = self.ae.quant_conv(h)
            posterior = DiagonalGaussianDistribution(moments)
            z = posterior.sample() if sample_posterior else posterior.mode()
            dec = self.ae.decoder(self.ae.post_quant_conv(z))
        finally:
            for hk in hooks:
                hk.remove()
        return dec, posterior

    def get_last_layer(self):
        return self.ae.get_last_layer()


print("QualityAwareAutoencoderKL defined (FiLM).")

## 3. Loading the two models

- **cVAE**: MedVAE fine-tuned on ARCADE **with** FiLM conditioning (notebook 4)
- **Baseline**: MedVAE fine-tuned on ARCADE **without** conditioning (same data, same compute budget, same hyperparameters — a clean ablation trained in NB04)

In [ ]:
# --- Quality-aware model (cVAE) ---
def load_cvae(ckpt_path: Path) -> QualityAwareAutoencoderKL:
    ckpt    = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    base_ae = AutoencoderKL2D(ddconfig=ckpt["ddconfig"], embed_dim=ckpt["embed_dim"],
                               apply_channel_ds=True)
    model   = QualityAwareAutoencoderKL(base_ae)
    model.load_state_dict(ckpt["model_state_dict"])
    return model, ckpt

if CVAE_CKPT.exists():
    cvae, ckpt_data = load_cvae(CVAE_CKPT)
    cvae = cvae.to(DEVICE).eval()
    print(f"cVAE       : {sum(p.numel() for p in cvae.parameters()):,} parameters")
    print(f"Checkpoint : {CVAE_CKPT.name}")
else:
    print("[!] Checkpoint not found — cold start (random weights).")
    base_ae2 = AutoencoderKL2D(ddconfig=DDCONFIG, embed_dim=EMBED_DIM, apply_channel_ds=True)
    cvae = QualityAwareAutoencoderKL(base_ae2).to(DEVICE).eval()
    ckpt_data = {}

# --- Baseline model: VAE fine-tuned on ARCADE WITHOUT conditioning (ablation) ---
# Loaded from baseline_state_dict: trained independently in NB04 with the
# same data, the same compute budget, the same hyperparameters — but WITHOUT FiLM.
# Any performance difference is then attributable to the conditioning c.
baseline = AutoencoderKL2D(
    ddconfig=DDCONFIG, embed_dim=EMBED_DIM, apply_channel_ds=True,
).to(DEVICE)

if "baseline_state_dict" in ckpt_data:
    baseline.load_state_dict(ckpt_data["baseline_state_dict"])
    bl_steps = ckpt_data.get("baseline_steps", "?")
    bl_val   = ckpt_data.get("best_val_loss_baseline", "?")
    print(f"Baseline   : VAE fine-tuned on ARCADE WITHOUT FiLM ({bl_steps} steps, val_loss={bl_val}) ✓")
elif CVAE_CKPT.exists():
    # Fallback: if the checkpoint was generated before the ablation was added,
    # use the pre-trained weights as a temporary baseline
    PRETRAINED_CKPT = Path("outputs_cvae/pretrained_weights/vae_4x_4c_2D.ckpt")
    if "pretrained_state_dict" in ckpt_data:
        baseline.load_state_dict(ckpt_data["pretrained_state_dict"])
        print("⚠ baseline_state_dict missing — falling back to the pre-trained weights.")
        print("  → Rerun notebook 4 to generate the real baseline (ablation).")
    elif PRETRAINED_CKPT.exists():
        baseline_fresh = AutoencoderKL2D(
            ddconfig=DDCONFIG, embed_dim=EMBED_DIM,
            ckpt_path=PRETRAINED_CKPT, apply_channel_ds=True,
        )
        baseline.load_state_dict(baseline_fresh.state_dict())
        del baseline_fresh
        print(f"⚠ baseline_state_dict missing — falling back to {PRETRAINED_CKPT.name}.")
        print("  → Rerun notebook 4 to generate the real baseline (ablation).")
    else:
        print("⚠ No baseline found. Rerun notebook 4.")
else:
    print("⚠ Checkpoint not found — baseline with random weights.")

baseline.eval()
print(f"Baseline   : {sum(p.numel() for p in baseline.parameters()):,} parameters")
print(f"\n→ Comparison: fine-tuned WITH c (cVAE+FiLM) vs fine-tuned WITHOUT c (ablation)")

## 4. Building the test sets

In [ ]:
from pipeline_config import read_csv_with_paths
df_labels = read_csv_with_paths(LABELS_CSV)

# Recompute quality_score/quality_label from the column of the
# active approach (instead of trusting the generic aliases of the CSV, which only reflect
# the last approach NB03 was run with): makes this notebook
# robust to the order in which A/B/C are run.
SCORE_COL = get_score_column()
if SCORE_COL not in df_labels.columns:
    raise ValueError(
        f"Column '{SCORE_COL}' missing from {LABELS_CSV} - rerun NB03 with "
        f"APPROACH='{APPROACH}' before this notebook."
    )
df_labels["quality_score"] = df_labels[SCORE_COL]
train_scores = df_labels.loc[df_labels["split"] == "train", SCORE_COL]
t33, t67 = train_scores.quantile(1 / 3), train_scores.quantile(2 / 3)
df_labels["quality_label"] = df_labels[SCORE_COL].apply(
    lambda s: "bad" if s < t33 else ("medium" if s < t67 else "good")
)

print(f"Dataset : {len(df_labels)} images  —  labels : {df_labels['quality_label'].value_counts().to_dict()}")

# Stratified selection
test_sets = {}
for label in ["bad", "medium", "good"]:
    subset = df_labels[df_labels["quality_label"] == label]
    # Check that the files are accessible
    accessible = subset[subset["path"].apply(lambda p: Path(p).exists())]
    sampled    = accessible.sample(min(N_TEST_PER_CLASS, len(accessible)),
                                    random_state=RANDOM_SEED)
    test_sets[label] = sampled.reset_index(drop=True)
    print(f"  {label:6s} : {len(sampled):3d} images accessibles")

df_test = pd.concat(test_sets.values(), ignore_index=True)
print(f"\nTotal : {len(df_test)} images de test")

## 5. Utility functions

### 5a. Loading and preprocessing

In [ ]:
def load_tensor(path: str, size: int = IMG_SIZE) -> torch.Tensor:
    """Load an image → FloatTensor (1, 1, H, W) in [-1, 1]."""
    img = Image.open(path).convert("L").resize((size, size), Image.Resampling.BILINEAR)
    t   = torch.from_numpy(np.asarray(img, np.float32) / 255.0)
    return (t.unsqueeze(0).unsqueeze(0) * 2.0 - 1.0)  # (1,1,H,W)


def to_uint8(t: torch.Tensor) -> np.ndarray:
    """Tensor [-1,1] (1,1,H,W) → np.uint8 (H,W) for the skimage metrics."""
    arr = t.squeeze().cpu().numpy()
    return ((arr.clip(-1, 1) + 1) / 2 * 255).astype(np.uint8)

### 5b. Reconstruction metrics

| Metric | Description | Direction |
|---|---|---|
| **MSE** | Pixel-wise mean squared error | ↓ better |
| **SSIM** | Structural similarity index | ↑ better |
| **PSNR** | Peak signal-to-noise ratio | ↑ better |
| **HaarPSI** | Haar perceptual similarity index (wavelets) | ↑ better |

> **Note on SSIM / PSNR:** although SSIM and PSNR are computed here by convention,
> recent studies (Breger et al., 2024 — *A study on the adequacy of common IQA measures for
> medical images*) show that these metrics are **weakly correlated with perceived quality**
> on cardiac and chest X-rays. **HaarPSI**, based on Haar wavelets, proved significantly more
> reliable in that study. Visual inspection and the MSE on the anatomy remain the primary
> indicators; HaarPSI serves as a perceptual check.

In [ ]:
def compute_metrics(original: torch.Tensor, reconstruction: torch.Tensor) -> dict:
    """
    original, reconstruction: (1, 1, H, W) in [-1, 1]
    Returns MSE, SSIM, PSNR, HaarPSI.
    """
    mse  = float(F.mse_loss(reconstruction, original).item())

    orig_u8 = to_uint8(original)
    rec_u8  = to_uint8(reconstruction)

    ssim_val = float(ssim_fn(orig_u8, rec_u8, data_range=255))
    psnr_val = float(psnr_fn(orig_u8, rec_u8, data_range=255))

    # HaarPSI — perceptual metric based on Haar wavelets,
    # more reliable than SSIM/PSNR on medical images (Breger et al., 2024)
    haarpsi_val = _compute_haarpsi(orig_u8, rec_u8)

    return {"mse": mse, "ssim": ssim_val, "psnr": psnr_val, "haarpsi": haarpsi_val}


def _compute_haarpsi(ref: np.ndarray, deg: np.ndarray) -> float:
    """
    Simplified HaarPSI with 2-scale Haar wavelets.
    Tries the `piq` library first, then falls back to numpy.
    """
    try:
        import piq
        # piq attend (B, C, H, W) float tensors
        r_t = torch.from_numpy(ref.astype(np.float32)).unsqueeze(0).unsqueeze(0)
        d_t = torch.from_numpy(deg.astype(np.float32)).unsqueeze(0).unsqueeze(0)
        return float(piq.haarpsi(r_t, d_t, data_range=255.).item())
    except (ImportError, Exception):
        pass

    # Fallback: numpy computation (2 Haar scales)
    from scipy.ndimage import convolve, zoom

    C = 3e-4  # stability constant (Reisenhofer et al., 2018)
    ref64 = ref.astype(np.float64) / 255.0
    deg64 = deg.astype(np.float64) / 255.0

    total_score, total_weight = 0.0, 0.0
    for scale in [1, 2]:
        if scale > 1:
            r = zoom(ref64, 0.5, order=1)
            d = zoom(deg64, 0.5, order=1)
        else:
            r, d = ref64, deg64

        for kernel in [np.array([[1, -1]], dtype=np.float64),
                       np.array([[1], [-1]], dtype=np.float64)]:
            cr = np.abs(convolve(r, kernel, mode="reflect"))
            cd = np.abs(convolve(d, kernel, mode="reflect"))
            # Local similarity
            sim = (2 * cr * cd + C) / (cr**2 + cd**2 + C)
            # Weights = saliency (max magnitude of the coefficients)
            w = np.maximum(cr, cd)
            total_score  += np.sum(w * sim)
            total_weight += np.sum(w)

    return float(total_score / (total_weight + 1e-10))


print("compute_metrics defined (MSE, SSIM, PSNR, HaarPSI).")

### 5c. Inline quality score (for synthetically degraded images)

In [ ]:
def inline_quality_score(img_tensor: torch.Tensor) -> float:
    """
    Compute a quality score ∈ [0,1] directly from a tensor.
    Uses Tenengrad + Laplacian variance after MSCN normalisation.
    Follows the logic of notebook 2 (reference: method A of notebook 3).

    A Gaussian pre-smoothing (σ=1 px) makes the score robust to noise:
    the structure is preserved but the high-frequency noise is attenuated first.
    A noise penalty term (high-frequency residual) prevents saturation at 1.0.
    """
    from scipy.ndimage import laplace, sobel, gaussian_filter

    img = img_tensor.squeeze().cpu().numpy().astype(np.float64)
    # Remap [-1,1] → [0,1]
    img = (img + 1) / 2

    # Pre-smoothing to attenuate the noise without destroying the structure
    img_smooth = gaussian_filter(img, sigma=1.0)

    # MSCN on the smoothed image
    mu    = uniform_filter(img_smooth, size=7)
    sigma = np.sqrt(np.maximum(uniform_filter(img_smooth**2, size=7) - mu**2, 0))
    mscn  = (img_smooth - mu) / (sigma + 1.0)

    lap_var   = float(laplace(mscn).var())
    Gx        = sobel(mscn, axis=1)
    Gy        = sobel(mscn, axis=0)
    tenengrad = float(np.sqrt(Gx**2 + Gy**2).mean())

    # Noise penalty: energy of the high-frequency residual (original − smoothed)
    noise_energy = float(np.mean((img - img_smooth) ** 2))
    noise_penalty = float(np.clip(1.0 - noise_energy / 0.005, 0.0, 1.0))

    raw_score = np.clip(0.5 * (lap_var / 0.01) + 0.5 * (tenengrad / 0.5), 0.0, 1.0)
    score = raw_score * noise_penalty
    return float(score)

# Check
if len(df_test) > 0:
    x_check = load_tensor(df_test["path"].iloc[0])
    print(f"Inline quality score (example): {inline_quality_score(x_check):.4f}")

## 6. Synthetic degradations

For a controlled benchmark, the best ARCADE images are artificially degraded.
This follows the methodology validated in paper 3 (Cryptosporidium IQA, 2026).

In [ ]:
def add_gaussian_noise(x: torch.Tensor, sigma: float) -> torch.Tensor:
    return (x + torch.randn_like(x) * sigma).clamp(-1, 1)


def add_motion_blur(x: torch.Tensor, kernel_size: int, direction: str = "horizontal") -> torch.Tensor:
    """Simulate cardiac motion blur (mainly vertical direction)."""
    if direction == "horizontal":
        k = torch.ones(1, 1, 1, kernel_size, device=x.device) / kernel_size
        pad = (kernel_size // 2, kernel_size // 2, 0, 0)
    else:
        k = torch.ones(1, 1, kernel_size, 1, device=x.device) / kernel_size
        pad = (0, 0, kernel_size // 2, kernel_size // 2)
    return F.conv2d(F.pad(x, pad, mode="reflect"), k).clamp(-1, 1)


DEGRADATION_GRID = [
    {"name": "Propre",          "fn": lambda x: x,                             "severity": 0.0},
    {"name": "Bruit σ=0.05",   "fn": lambda x: add_gaussian_noise(x, 0.05),   "severity": 0.05},
    {"name": "Bruit σ=0.10",   "fn": lambda x: add_gaussian_noise(x, 0.10),   "severity": 0.10},
    {"name": "Bruit σ=0.20",   "fn": lambda x: add_gaussian_noise(x, 0.20),   "severity": 0.20},
    {"name": "Flou H k=5",     "fn": lambda x: add_motion_blur(x, 5,  "horizontal"), "severity": 5},
    {"name": "Flou V k=5",     "fn": lambda x: add_motion_blur(x, 5,  "vertical"),   "severity": 5},
    {"name": "Flou V k=11",    "fn": lambda x: add_motion_blur(x, 11, "vertical"),   "severity": 11},
]

# Grid of degradations on a test image
if len(test_sets.get("good", pd.DataFrame())) > 0:
    x_demo = load_tensor(test_sets["good"]["path"].iloc[0]).to(DEVICE)

    fig, axes = plt.subplots(1, len(DEGRADATION_GRID), figsize=(3 * len(DEGRADATION_GRID), 3.5))
    for ax, d in zip(axes, DEGRADATION_GRID):
        with torch.no_grad():
            x_deg = d["fn"](x_demo)
        ax.imshow(to_uint8(x_deg), cmap="gray", vmin=0, vmax=255)
        ax.set_title(d["name"], fontsize=8)
        ax.axis("off")
    fig.suptitle("Grid of synthetic degradations", fontsize=11)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "05_degradation_grid.png", dpi=120, bbox_inches="tight")
    plt.show()

## 7. Quantitative evaluation — real images

Both models are evaluated on the `bad/medium/good` images of the test set.
The cVAE uses the real `quality_score` of each image.

In [ ]:
@torch.no_grad()
def evaluate_on_dataframe(df: pd.DataFrame, label: str) -> pd.DataFrame:
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"{label:6s}", leave=False):
        try:
            x = load_tensor(row["path"]).to(DEVICE)
            c = torch.tensor([row["quality_score"]], dtype=torch.float32, device=DEVICE)

            # Baseline: standard forward pass (no conditioning)
            rec_base, post_base, _ = baseline(x, sample_posterior=False, decode=True)
            m_base = compute_metrics(x, rec_base)

            # cVAE: forward pass conditioned on the true quality score
            rec_cvae, post_cvae = cvae(x, c, sample_posterior=False)
            m_cvae = compute_metrics(x, rec_cvae)

            records.append({
                "image_id":      row["image_id"],
                "path":          row["path"],
                "quality_label": label,
                "quality_score": row["quality_score"],
                **{f"base_{k}": v for k, v in m_base.items()},
                **{f"cvae_{k}": v for k, v in m_cvae.items()},
            })
        except Exception as e:
            pass  # file not accessible
    return pd.DataFrame(records)


results_per_label = {}
for label, df_subset in test_sets.items():
    results_per_label[label] = evaluate_on_dataframe(df_subset, label)

df_results = pd.concat(results_per_label.values(), ignore_index=True)
print(f"\nEvaluation done: {len(df_results)} images")
df_results.groupby("quality_label")[["base_mse", "cvae_mse", "base_ssim", "cvae_ssim", "base_haarpsi", "cvae_haarpsi"]].mean().round(5)

## 8. Evaluation on synthetically degraded images

Starting from the best images (known ground truth), the reconstruction MSE is measured
against the **original clean image** — not the degraded one.

In [ ]:
@torch.no_grad()
def evaluate_synthetic(df_good: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in tqdm(df_good.iterrows(), total=len(df_good), desc="Synthetic", leave=False):
        try:
            x_clean = load_tensor(row["path"]).to(DEVICE)
        except Exception:
            continue

        for d in DEGRADATION_GRID:
            x_deg = d["fn"](x_clean)
            c_deg = torch.tensor([inline_quality_score(x_deg)], device=DEVICE)

            rec_base, _, _ = baseline(x_deg, sample_posterior=False, decode=True)
            rec_cvae, _    = cvae(x_deg, c_deg, sample_posterior=False)

            # Metrics vs the CLEAN image (ground truth)
            m_base = compute_metrics(x_clean, rec_base)
            m_cvae = compute_metrics(x_clean, rec_cvae)

            records.append({
                "image_id":    row["image_id"],
                "degradation": d["name"],
                "severity":    d["severity"],
                "c_inline":    float(c_deg.item()),
                **{f"base_{k}": v for k, v in m_base.items()},
                **{f"cvae_{k}": v for k, v in m_cvae.items()},
            })
    return pd.DataFrame(records)


df_synth = evaluate_synthetic(test_sets.get("good", pd.DataFrame()))
print(f"Synthetic images evaluated: {len(df_synth)} entries")
if len(df_synth):
    df_synth.groupby("degradation")[["base_mse", "cvae_mse", "base_ssim", "cvae_ssim", "base_haarpsi", "cvae_haarpsi"]].mean().round(5)

In [ ]:
# Plot: reconstruction MSE vs degradation type
if len(df_synth) > 0:
    order   = [d["name"] for d in DEGRADATION_GRID]
    grouped = df_synth.groupby("degradation")[["base_mse", "cvae_mse"]].mean().reindex(order)

    x_pos = np.arange(len(order))
    width = 0.35
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, metric, ylabel in zip(
        axes,
        [["base_mse", "cvae_mse"], ["base_ssim", "cvae_ssim"], ["base_haarpsi", "cvae_haarpsi"]],
        ["MSE ↓ (lower is better)", "SSIM ↑ (higher is better)", "HaarPSI ↑ (higher is better)"],
    ):
        g = df_synth.groupby("degradation")[metric].mean().reindex(order)
        bars1 = ax.bar(x_pos - width/2, g[metric[0]], width, label="Baseline",   color="steelblue",   alpha=0.85)
        bars2 = ax.bar(x_pos + width/2, g[metric[1]], width, label="cVAE (ours)", color="darkorange", alpha=0.85)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(order, rotation=30, ha="right", fontsize=8)
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.set_title(ylabel.split(" ")[0])
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle("Reconstruction vs ground truth — synthetically degraded images", fontsize=11)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "05_synthetic_metrics.png", dpi=120, bbox_inches="tight")
    plt.show()

## 9. Statistical comparison

Test whether the performance difference is **statistically significant** on the `bad` images.
Wilcoxon test (non-parametric, robust for small samples).

In [ ]:
for metric, direction in [("mse", "↓ lower is better"), ("ssim", "↑ higher is better"), ("psnr", "↑ higher is better"), ("haarpsi", "↑ higher is better")]:
    print(f"\n{'='*60}")
    print(f"Metric: {metric.upper()} ({direction})")
    print(f"{'='*60}")
    print(f"{'Label':8s}  {'Baseline':>12s}  {'cVAE':>12s}  {'Δ':>10s}  {'p-value':>10s}")

    for label in ["bad", "medium", "good"]:
        sub = df_results[df_results["quality_label"] == label]
        if len(sub) < 2:
            continue
        base_vals = sub[f"base_{metric}"].values
        cvae_vals = sub[f"cvae_{metric}"].values

        mean_b = np.mean(base_vals)
        mean_c = np.mean(cvae_vals)
        delta  = mean_c - mean_b

        try:
            stat, pval = stats.wilcoxon(base_vals, cvae_vals)
        except Exception:
            pval = float("nan")

        sig = "✓ sig." if (not np.isnan(pval) and pval < 0.05) else "—"
        print(f"{label:8s}  {mean_b:12.5f}  {mean_c:12.5f}  {delta:+10.5f}  {pval:10.4f}  {sig}")

In [ ]:
# MSE and SSIM box plots per quality class
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

label_order  = ["bad", "medium", "good"]
label_colors = {"bad": "crimson", "medium": "darkorange", "good": "forestgreen"}

for ax, metric, title, better in zip(
    axes,
    ["mse",  "ssim",  "haarpsi"],
    ["MSE ↓", "SSIM ↑", "HaarPSI ↑"],
    ["lower", "higher", "higher"],
):
    positions_b, positions_c = [], []
    data_b, data_c           = [], []

    for i, label in enumerate(label_order):
        sub = df_results[df_results["quality_label"] == label]
        if len(sub) == 0:
            continue
        data_b.append(sub[f"base_{metric}"].values)
        data_c.append(sub[f"cvae_{metric}"].values)
        positions_b.append(i * 3)
        positions_c.append(i * 3 + 1)

    if data_b:
        bp1 = ax.boxplot(data_b, positions=positions_b, widths=0.7,
                         patch_artist=True, boxprops=dict(facecolor="steelblue", alpha=0.6),
                         medianprops=dict(color="navy", lw=2))
        bp2 = ax.boxplot(data_c, positions=positions_c, widths=0.7,
                         patch_artist=True, boxprops=dict(facecolor="darkorange", alpha=0.7),
                         medianprops=dict(color="saddlebrown", lw=2))

    ax.set_xticks([i * 3 + 0.5 for i in range(len(label_order))])
    ax.set_xticklabels(label_order)
    ax.set_title(title, fontsize=11)
    ax.grid(axis="y", alpha=0.3)

    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor="steelblue", label="Baseline"),
                       Patch(facecolor="darkorange", label="cVAE")], fontsize=9)

fig.suptitle("Distribution of the metrics per quality class", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_boxplots.png", dpi=120, bbox_inches="tight")
plt.show()

## 10. Qualitative gallery

Main figure of the report: **Original | Baseline | cVAE** on the most degraded images.

In [ ]:
@torch.no_grad()
def build_gallery(df_sub: pd.DataFrame, n: int = 5, title: str = ""):
    rows = df_sub.sort_values("quality_score").head(n)

    fig, axes = plt.subplots(n, 4, figsize=(4 * 4, n * 3.5))
    if n == 1:
        axes = axes[np.newaxis, :]

    col_titles = ["Original", "Baseline", "cVAE (ours)", "Diff (Baseline − cVAE)"]
    for ax, ct in zip(axes[0], col_titles):
        ax.set_title(ct, fontsize=9, fontweight="bold")

    for row_idx, (_, record) in enumerate(rows.iterrows()):
        x = load_tensor(record["path"]).to(DEVICE)
        c = torch.tensor([record["quality_score"]], device=DEVICE)

        rec_base, _, _ = baseline(x, sample_posterior=False, decode=True)
        rec_cvae, _    = cvae(x, c, sample_posterior=False)

        orig_np  = to_uint8(x)
        base_np  = to_uint8(rec_base)
        cvae_np  = to_uint8(rec_cvae)
        diff_np  = base_np.astype(np.float32) - cvae_np.astype(np.float32)

        m_base = compute_metrics(x, rec_base)
        m_cvae = compute_metrics(x, rec_cvae)

        for col, (data, cmap, vmin, vmax) in enumerate([
            (orig_np, "gray", 0, 255),
            (base_np, "gray", 0, 255),
            (cvae_np, "gray", 0, 255),
            (diff_np, "RdBu_r", -60, 60),
        ]):
            axes[row_idx, col].imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
            axes[row_idx, col].axis("off")

        axes[row_idx, 0].set_ylabel(f"c={record['quality_score']:.3f}", fontsize=8)
        axes[row_idx, 1].set_xlabel(
            f"MSE={m_base['mse']:.4f}  SSIM={m_base['ssim']:.3f}  HP={m_base['haarpsi']:.3f}", fontsize=7, color="steelblue")
        axes[row_idx, 2].set_xlabel(
            f"MSE={m_cvae['mse']:.4f}  SSIM={m_cvae['ssim']:.3f}  HP={m_cvae['haarpsi']:.3f}", fontsize=7, color="darkorange")

    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    return fig


# Gallery on the bad images (where the effect of the cVAE should be most visible)
if len(results_per_label.get("bad", pd.DataFrame())) >= 3:
    fig_gal = build_gallery(
        results_per_label["bad"],
        n=min(5, len(results_per_label["bad"])),
        title="Gallery — degraded images (bad): baseline vs quality-aware cVAE"
    )
    fig_gal.savefig(FIGURES_DIR / "05_gallery_bad.png", dpi=120, bbox_inches="tight")
    plt.show()

In [ ]:
# Gallery on the synthetically degraded images (controlled benchmark)
if len(test_sets.get("good", pd.DataFrame())) > 0:
    x_gt = load_tensor(test_sets["good"]["path"].iloc[0]).to(DEVICE)

    fig, axes = plt.subplots(3, len(DEGRADATION_GRID), figsize=(3 * len(DEGRADATION_GRID), 10))
    row_labels = ["Degraded (input)", "Baseline", "cVAE"]

    for col, d in enumerate(DEGRADATION_GRID):
        x_deg = d["fn"](x_gt)
        c_deg = torch.tensor([inline_quality_score(x_deg)], device=DEVICE)

        with torch.no_grad():
            rec_b, _, _ = baseline(x_deg, sample_posterior=False, decode=True)
            rec_c, _    = cvae(x_deg, c_deg, sample_posterior=False)

        for row_idx, (arr, cmap) in enumerate([
            (to_uint8(x_deg), "gray"),
            (to_uint8(rec_b),  "gray"),
            (to_uint8(rec_c),  "gray"),
        ]):
            axes[row_idx, col].imshow(arr, cmap=cmap, vmin=0, vmax=255)
            axes[row_idx, col].axis("off")
            if col == 0:
                axes[row_idx, col].set_ylabel(row_labels[row_idx], fontsize=8)
        axes[0, col].set_title(d["name"] + f"\nc={float(c_deg):.2f}", fontsize=7)

    fig.suptitle("Reconstruction of synthetically degraded images", fontsize=11)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "05_gallery_synthetic.png", dpi=120, bbox_inches="tight")
    plt.show()

## 11. Summary table

In [ ]:
print("=" * 75)
print(f"{'TABLEAU RÉCAPITULATIF':^75}")
print("=" * 75)

for metric, better in [("mse", "↓"), ("ssim", "↑"), ("psnr", "↑")]:
    print(f"\n  {metric.upper()} {better}")
    print(f"  {'Label':8s}  {'Baseline':>12s}  {'cVAE':>12s}  {'Δ (cvae−base)':>15s}  {'Improvement':>14s}")
    print("  " + "-" * 65)
    for label in ["bad", "medium", "good", "ALL"]:
        if label == "ALL":
            sub = df_results
        else:
            sub = df_results[df_results["quality_label"] == label]
        if len(sub) == 0:
            continue
        mb = sub[f"base_{metric}"].mean()
        mc = sub[f"cvae_{metric}"].mean()
        d  = mc - mb
        # For MSE: improvement if Δ < 0; for SSIM/PSNR: if Δ > 0
        improved = (d < 0) if metric == "mse" else (d > 0)
        marker = "✓" if improved else "✗"
        pct = abs(d) / (abs(mb) + 1e-12) * 100
        print(f"  {label:8s}  {mb:12.5f}  {mc:12.5f}  {d:+15.5f}  {marker} {pct:+.1f}%")

print("\n" + "=" * 75)

## 12. Exporting the results

In [ ]:
# Results on real images
df_results.to_csv(TABLES_DIR / "05_evaluation_results.csv", index=False)
print(f"Saved → {TABLES_DIR / '05_evaluation_results.csv'}")

# Results on synthetic degradations
if len(df_synth) > 0:
    df_synth.to_csv(TABLES_DIR / "05_synthetic_results.csv", index=False)
    print(f"Saved → {TABLES_DIR / '05_synthetic_results.csv'}")

## Summary of notebook 5

| Experiment | Description | Figure |
|---|---|---|
| Degradation grid | 7 types of synthetic degradations | `05_degradation_grid.png` |
| Real metrics | MSE/SSIM/PSNR per bad/medium/good class | `05_boxplots.png` |
| Synthetic benchmark | MSE vs clean ground truth | `05_synthetic_metrics.png` |
| Wilcoxon test | Statistical significance on the bad images | Console |
| Bad gallery | Original \| Baseline \| cVAE \| Diff | `05_gallery_bad.png` |
| Synthetic gallery | Degraded input \| Baseline \| cVAE | `05_gallery_synthetic.png` |

---

### Ablation protocol

The **baseline** is a VAE trained **independently** in NB04 with:
- the **same data** (ARCADE, same train/val split)
- the **same compute budget** (N_WARMUP_STEPS + N_FINETUNE_STEPS = 7,500 steps)
- the **same hyperparameters** (LR=1e-6, cosine annealing, weight_decay=1e-4, gradient clipping=1.0)
- the **same architecture** (AutoencoderKL2D, identical ddconfig)
- **no FiLM**: no conditioning on the quality score $c$

This is meant as a **rigorous ablation**: any performance difference between the baseline and the
cVAE is **attributable only to the FiLM conditioning on $c$**, not to the effect of fine-tuning on
ARCADE.

### Observed results

See the cell below — the conclusions are computed from the evaluation results.

In [ ]:
# ── Observed results — computed from the evaluation ──
from pipeline_config import APPROACH, get_approach_config
from scipy.stats import wilcoxon

cfg = get_approach_config()
print(f"Approach: {APPROACH} — {cfg['name']}")
print(f"Score used: {cfg['score_col']}")
print("=" * 70)

# Metrics to analyse (direction: -1 = lower is better, +1 = higher is better)
eval_metrics = {
    "MSE":     ("base_mse",     "cvae_mse",     -1),
    "SSIM":    ("base_ssim",    "cvae_ssim",     1),
    "PSNR":    ("base_psnr",    "cvae_psnr",     1),
    "HAARPSI": ("base_haarpsi", "cvae_haarpsi",  1),
}

labels = ["bad", "medium", "good"]
n_baseline_wins = 0
n_cvae_wins = 0
n_tests = 0
worst_class = None
worst_pct = 0

for metric, (col_b, col_c, direction) in eval_metrics.items():
    if col_b not in df_results.columns or col_c not in df_results.columns:
        continue
    arrow = "↓" if direction == -1 else "↑"
    print(f"\n{metric} ({arrow} better)")
    print("-" * 60)
    for label in labels:
        sub = df_results[df_results["quality_label"] == label]
        if len(sub) < 3:
            continue
        mean_b = sub[col_b].mean()
        mean_c = sub[col_c].mean()
        delta = mean_c - mean_b
        pct = abs(delta / mean_b) * 100 if mean_b != 0 else 0

        # Wilcoxon signed-rank test
        try:
            stat, pval = wilcoxon(sub[col_b], sub[col_c])
            sig = "sig." if pval < 0.05 else "n.s."
        except Exception:
            pval, sig = float("nan"), "?"

        # Who wins?
        if direction == -1:  # lower is better
            winner = "baseline" if mean_b < mean_c else "cVAE"
        else:  # higher is better
            winner = "baseline" if mean_b > mean_c else "cVAE"

        if winner == "baseline":
            n_baseline_wins += 1
        else:
            n_cvae_wins += 1
        n_tests += 1

        # Track worst class for MSE
        if metric == "MSE" and pct > worst_pct:
            worst_pct = pct
            worst_class = label

        print(f"  {label:<8s}  base={mean_b:.5f}  cvae={mean_c:.5f}  "
              f"Δ={delta:+.5f} ({pct:.1f}%)  p={pval:.4f} {sig}  → {winner}")

# Conclusion dynamique
print("\n" + "=" * 70)
print("CONCLUSION")
print("=" * 70)

if n_baseline_wins > n_cvae_wins:
    print(f"The vanilla baseline is better in {n_baseline_wins}/{n_tests} comparisons.")
    print(f"FiLM conditioning on the score c (approach {APPROACH}) "
          f"does not improve the reconstruction.")
    if worst_class:
        print(f"The largest MSE gap is on the '{worst_class}' images ({worst_pct:.1f}%).")
elif n_cvae_wins > n_baseline_wins:
    print(f"The conditioned cVAE is better in {n_cvae_wins}/{n_tests} comparisons.")
    print(f"FiLM conditioning on the score c (approach {APPROACH}) "
          f"improves the reconstruction.")
else:
    print(f"Mixed results: {n_baseline_wins} wins for the baseline, "
          f"{n_cvae_wins} wins for the cVAE out of {n_tests} comparisons.")